# TensorFlow / Keras Pipeline -- Sonar Classification

In this exercise we build an end-to-end binary classification pipeline with the **Keras 3** API on the **TensorFlow** backend. It is the Keras alternative to the PyTorch Sonar lab.

**Dataset:** `sonar.all-data.csv` -- 208 samples, 60 continuous sonar features, label column is `R` (rock) or `M` (mine).

**Evaluation protocol (as in the PyTorch lab):** three partitions are made before any scaling -- 124 training, 42 validation, and 42 test rows, with the same label mapping (Rock = 0, Mine = 1), split calls, and seed as the PyTorch notebook. The scaler is fitted on the training rows only. Early stopping and the architecture comparison use the validation rows; the test rows are used once, at the end, for the chosen model.

**Requirements:** TensorFlow 2.16 or later, which installs Keras 3 (for example `pip install "tensorflow>=2.16,<3"`, in a separate environment if it conflicts with PyTorch).

**Learning objectives:**
1. Build a Keras `Sequential` model and compile it with a loss that matches its output layer.
2. Use `model.fit()` with `EarlyStopping` and an explicit `validation_data` set.
3. Plot Keras training history (loss and accuracy curves).
4. Compare architectures on validation data, then evaluate the chosen model once on the test set.

---
## 1. Imports

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")   # must be set before importing keras

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras import layers, callbacks

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
print(f"Keras {keras.__version__}, backend: {keras.config.backend()}")

---
## 2. Load and Explore the Data

In [ ]:
col_names = [f"feat_{i}" for i in range(60)] + ["label"]
df = pd.read_csv("sonar.all-data.csv", header=None, names=col_names)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df["label"].value_counts()

---
## 3. Encode Labels, Split, and Standardize

We map the labels explicitly (Rock = 0, Mine = 1, the positive class) and print the mapping. Then we make three partitions **before** fitting the scaler, with the same calls and seed as the PyTorch lab (124 training, 42 validation, 42 test rows), and fit the scaler on the training rows only. Keras's `validation_split` is not used: it would take the *last* rows of the array without shuffling, after the scaler had already seen them.

In [ ]:
LABEL_MAP = {"R": 0, "M": 1}          # Rock = 0, Mine = 1 (positive class), as in the PyTorch lab
CLASS_NAMES = ["Rock", "Mine"]
y = df["label"].map(LABEL_MAP).to_numpy(dtype=np.float32)
X = df.drop(columns=["label"]).to_numpy(dtype=np.float32)

print(f"Label mapping: {LABEL_MAP}")
print(f"Class counts: Rock = {int((y == 0).sum())}, Mine = {int((y == 1).sum())}")

In [ ]:
# Three partitions before fitting the scaler (same calls and seed as the PyTorch lab)
dev_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.2,
                                     stratify=y, random_state=SEED)
train_idx, val_idx = train_test_split(dev_idx, test_size=0.25,
                                      stratify=y[dev_idx], random_state=SEED)
assert not set(train_idx) & set(val_idx) and not set(dev_idx) & set(test_idx)

scaler = StandardScaler().fit(X[train_idx])       # training rows only
X_train, X_val, X_test = (scaler.transform(X[idx]) for idx in (train_idx, val_idx, test_idx))
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

print(f"Train: {len(train_idx)}, validation: {len(val_idx)}, test: {len(test_idx)} (used once, section 10)")

---
## 4. Build the Keras Sequential Model

A two-hidden-layer network (the PyTorch lab uses one hidden layer of 32 units; translating that exact model is an extension):
- Dense(128, relu) + Dropout(0.3)
- Dense(64, relu) + Dropout(0.3)
- Dense(1, sigmoid): the output is a probability, not a logit

In [ ]:
keras.utils.set_random_seed(SEED)     # seeds Python, NumPy and the backend

model = keras.Sequential([
    keras.Input(shape=(60,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

model.summary()

---
## 5. Compile the Model

The output layer already applies a sigmoid, so the loss must expect probabilities: binary cross-entropy with `from_logits=False` (the default). With a linear output layer you would use `from_logits=True` instead, as on the slides; mixing a sigmoid output with `from_logits=True` applies the sigmoid twice.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.BinaryCrossentropy(from_logits=False),   # sigmoid output -> probabilities
    metrics=["accuracy"],
)

---
## 6. Train with EarlyStopping

`EarlyStopping` monitors the loss on the explicit validation rows passed with `validation_data`; `restore_best_weights=True` keeps the weights from the epoch with the lowest validation loss (Keras's copied checkpoint).

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1,
)
best_epoch = int(np.argmin(history.history["val_loss"])) + 1
print(f"Lowest validation loss at epoch {best_epoch} of {len(history.history['val_loss'])}")

---
## 7. Plot Training History

In [ ]:
epochs = range(1, len(history.history["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(epochs, history.history["loss"], label="Train Loss")
axes[0].plot(epochs, history.history["val_loss"], label="Val Loss")
axes[0].axvline(best_epoch, color="black", linestyle="--", label="Restored epoch")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, history.history["accuracy"], label="Train Acc")
axes[1].plot(epochs, history.history["val_accuracy"], label="Val Acc")
axes[1].axvline(best_epoch, color="black", linestyle="--", label="Restored epoch")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Evaluate on the Validation Set

Diagnose the restored model on the validation rows: loss, accuracy, classification report, and confusion matrix. These rows also chose the restored epoch, so the scores are slightly optimistic. With 42 rows, one row moves accuracy by 2.4 points.

In [ ]:
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_acc:.4f} ({round(val_acc * len(y_val))} of {len(y_val)})")

val_prob = model.predict(X_val, verbose=0).ravel()       # probabilities (sigmoid output)
val_pred = (val_prob >= 0.5).astype(int)

print("\n" + classification_report(y_val, val_pred, target_names=CLASS_NAMES))
print("Confusion matrix (rows = true, columns = predicted):")
print(confusion_matrix(y_val, val_pred))

---
## 9. Compare Architectures on the Validation Set

Each candidate is trained on the same training rows with early stopping on the validation rows, and compared by its validation loss at the restored epoch; we choose the lowest. The test set plays no part in the choice. The chosen score is optimistic (it was selected on these 42 rows), and a difference of one or two rows is not evidence.

In [ ]:
def build_model(hidden_units, dropout_rate=0.3, lr=1e-3):
    """Build and compile a Keras model with the given hidden layer sizes."""
    model = keras.Sequential([keras.Input(shape=(60,))])
    for units in hidden_units:
        model.add(layers.Dense(units, activation="relu"))
        model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.BinaryCrossentropy(from_logits=False),
        metrics=["accuracy"],
    )
    return model


architectures = [
    {"hidden_units": [128, 64], "dropout_rate": 0.3, "lr": 1e-3},
    {"hidden_units": [64, 32], "dropout_rate": 0.2, "lr": 1e-3},
    {"hidden_units": [256, 128, 64], "dropout_rate": 0.4, "lr": 5e-4},
]

results, candidates = [], []
for cfg in architectures:
    keras.utils.set_random_seed(SEED)
    m = build_model(**cfg)
    es = callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=150,
              batch_size=16, callbacks=[es], verbose=0)
    loss, acc = m.evaluate(X_val, y_val, verbose=0)
    results.append({**cfg, "restored_epoch": int(np.argmin(h.history["val_loss"])) + 1,
                    "val_loss": round(loss, 4), "val_acc": round(acc, 4)})
    candidates.append(m)
    print(f"{cfg} => Val Loss: {loss:.4f}, Val Acc: {acc:.4f}")

results = pd.DataFrame(results)
best_i = int(results["val_loss"].idxmin())
final_model = candidates[best_i]
print(f"\nChosen on validation loss: {architectures[best_i]}")
results

---
## 10. One Locked Final Assessment

The architecture was chosen on the validation rows. Evaluate the chosen model **once** on the 42 test rows and report the score with its count. Do not change the architecture because of this result: that would turn the test set into a second validation set.

In [ ]:
# The only use of the test set: the architecture chosen on validation, evaluated once
test_loss, test_acc = final_model.evaluate(X_test, y_test, verbose=0)
print(f"Chosen architecture: {architectures[best_i]}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f} ({round(test_acc * len(y_test))} of {len(y_test)})")

y_prob = final_model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("\n" + classification_report(y_test, y_pred, target_names=CLASS_NAMES))
print("Confusion matrix (rows = true, columns = predicted):")
print(confusion_matrix(y_test, y_pred))

---
## 11. Summary

In this notebook we:

1. Loaded the Sonar dataset and mapped the labels explicitly (Rock = 0, Mine = 1).
2. Made training, validation, and test partitions before scaling, as in the PyTorch lab, and fitted the scaler on the training rows only.
3. Built a Keras `Sequential` model with two hidden layers and a sigmoid output.
4. Compiled it with Adam and binary cross-entropy on probabilities (`from_logits=False`).
5. Trained with `model.fit()`, explicit `validation_data`, and `EarlyStopping` with restored best weights.
6. Plotted training history (loss and accuracy curves).
7. Compared architectures on the validation rows and evaluated the chosen one once on the test rows.

**Key takeaways:**
- Keras provides a high-level API that abstracts away the training loop.
- Pass explicit `validation_data`; `validation_split` takes the last rows unshuffled, after preprocessing has already seen them.
- `EarlyStopping(restore_best_weights=True)` is Keras's copied checkpoint.
- Choose with validation data, then touch the test set once.